# Build your own model quantizer



**Real-world constraint:** distribute the model to laptops using **at least 45% less checkpoint storage than FP32**, with **at most 1.0 percentage point of SST-2 validation accuracy loss**.

**Experiments:** FP32 baseline, then **symmetric and asymmetric INT8**, each using **per-tensor**, **per-channel**, and **per-group (64)**. That is six quantized configurations. We compare both grid placement and how many weights share a scale; neither mode is automatically better.



## 2. Imports and experiment settings

We use PyTorch for our own quantizer, Transformers only for the existing model/tokenizer, and Datasets for labelled examples. FP32 works simply on CPU. A fixed seed, batch size, token length and thread count make the comparison easier to reproduce.

Keep `EVAL_LIMIT = None` for the final report: that evaluates **all 872 validation examples**. You may temporarily set it to 128 for a quick practice run, but label those results as preliminary. The 45%/1-point limits are declared before seeing results.


In [ ]:
import copy
import gc
import json
import platform
import shutil
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
import transformers
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
GROUP_SIZE = 64
MAX_LENGTH = 128
BATCH_SIZE = 16
TIMING_BATCH_SIZE = 8
WARMUP_RUNS = 2
TIMING_RUNS = 7
EVAL_LIMIT = None
MIN_REDUCTION_PERCENT = 45.0
MAX_ACCURACY_DROP_PP = 1.0

torch.manual_seed(42)
torch.set_num_threads(2)
ARTIFACTS = Path("artifacts")
RESULTS = Path("results")
ARTIFACTS.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
print("PyTorch:", torch.__version__, "| device: CPU | threads:", torch.get_num_threads())


PyTorch: 2.5.1+cpu | device: CPU | threads: 2


## 3. Load the pretrained FP32 baseline

The model has already learned sentiment classification; no training is needed. `eval()` disables training-only behaviour such as dropout. We keep this original model unchanged so every quantized configuration starts from the same weights. The final layer produces two scores: negative and positive.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
baseline = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, attn_implementation="eager"
).float().cpu().eval()
parameter_count = sum(parameter.numel() for parameter in baseline.parameters())
print(f"Parameters: {parameter_count:,} ({parameter_count / 1e6:.1f} million)")
print("Labels:", baseline.config.id2label)
print("Baseline weight dtype:", next(baseline.parameters()).dtype)


Parameters: 66,955,010 (67.0 million)
Labels: {0: 'NEGATIVE', 1: 'POSITIVE'}
Baseline weight dtype: torch.float32


## 4. Prepare a fair evaluation dataset

We use the **SST-2 validation split of GLUE** because it has labels; the public test labels are unavailable. Accuracy is the fraction of correct positive/negative predictions. Tokenize once and reuse the exact same batches for every configuration.

For timing we use a fixed batch padded to 128 tokens. This measures the model forward pass, including our weight reconstruction, with an identical shape for every model. Tokenization is excluded from that timing.


In [4]:
validation = load_dataset("nyu-mll/glue", "sst2", split="validation").shuffle(seed=42)
if EVAL_LIMIT is not None:
    validation = validation.select(range(min(EVAL_LIMIT, len(validation))))

evaluation_batches = []
for start in range(0, len(validation), BATCH_SIZE):
    examples = validation[start:start + BATCH_SIZE]
    inputs = tokenizer(
        examples["sentence"], padding=True, truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt"
    )
    evaluation_batches.append((inputs, torch.tensor(examples["label"])))

timing_inputs = tokenizer(
    validation[:TIMING_BATCH_SIZE]["sentence"], padding="max_length",
    truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
)
print("Evaluation examples:", len(validation))
print("Timing shape [examples, tokens]:", tuple(timing_inputs["input_ids"].shape))


Evaluation examples: 872
Timing shape [examples, tokens]: (8, 128)


## 5. Quantize a matrix: symmetric and asymmetric

Both modes use raw PyTorch. Granularity decides which weights share a scale: the whole matrix, one output row, or a group of 64 weights.

**Symmetric:** `scale = max(abs(W))/127`, `q = clamp(round(W/scale), -127, 127)`. Zero point is 0 and does not need to be stored.

**Asymmetric:** use the full signed INT8 range `q_min = -128`, `q_max = 127`:

```text
w_min = min(min(weights), 0)
w_max = max(max(weights), 0)
scale = (w_max - w_min) / (q_max - q_min)
raw_zero_point = round(q_min - w_min / scale)
zero_point = clamp(raw_zero_point, q_min, q_max)
q = clamp(round(weight / scale) + zero_point, q_min, q_max)
```

**Boundary rule you requested:** if the calculated zero point exceeds **127**, cap it at **127**. If it is below **−128**, cap it at **−128**. These are integer-code bounds, not the smallest/largest original weight. Clamp before converting to INT8 so it cannot wrap around.

**Why include real zero in the weight range?** For positive-only values `[2, 4]`, using that range directly gives scale `2/255` and raw zero point `−383`. Clipping it to `−128` alone would make the largest reconstructable value just **2**, clipping the weight 4. Expanding the range to `[0, 4]` gives a suitable scale and zero point `−128`, preserving both weights approximately. A negative-only range similarly expands to include zero. With this expansion, the raw zero point should already be within the bounds; the explicit clamp enforces the rule.

All-zero regions use a safe scale of 1. Group padding is removed from stored weights. Asymmetric mode stores an extra INT8 zero point per scale; its bytes count in model size.


In [ ]:
def quantize_weight(weight, granularity="per_channel", group_size=64, mode="symmetric"):
    """Return INT8 weights, FP32 scales, and optional INT8 zero points."""
    if weight.ndim != 2:
        raise ValueError("Expected [output_features, input_features].")
    if granularity not in {"per_tensor", "per_channel", "per_group"}:
        raise ValueError("Choose per_tensor, per_channel, or per_group.")
    if not isinstance(group_size, int) or group_size <= 0:
        raise ValueError("group_size must be a positive integer.")
    if mode not in {"symmetric", "asymmetric"}:
        raise ValueError("Choose symmetric or asymmetric.")

    weight = weight.detach().to(torch.float32)
    rows, columns = weight.shape

    # View the data so each scale covers the requested region.
    if granularity == "per_tensor":
        values = weight
        minimum, maximum = values.amin(), values.amax()
    elif granularity == "per_channel":
        values = weight
        minimum = values.amin(dim=1, keepdim=True)  # [rows, 1]
        maximum = values.amax(dim=1, keepdim=True)
    else:
        padding = (-columns) % group_size
        values = F.pad(weight, (0, padding)).reshape(rows, -1, group_size)
        minimum = values.amin(dim=2, keepdim=True)  # [rows, groups, 1]
        maximum = values.amax(dim=2, keepdim=True)

    if mode == "symmetric":
        max_abs = torch.maximum(minimum.abs(), maximum.abs())
        scale = torch.where(max_abs > 0, max_abs / 127.0, torch.ones_like(max_abs))
        zero_point = None 
        integers = torch.round(values / scale).clamp(-127, 127).to(torch.int8)
    else:
        q_min, q_max = -128, 127
        # Include real zero BEFORE finding the scale. Just clipping a zero point
        # with the old scale could clip positive-only or negative-only weights.
        minimum = torch.minimum(minimum, torch.zeros_like(minimum))
        maximum = torch.maximum(maximum, torch.zeros_like(maximum))
        spread = maximum - minimum
        scale = torch.where(spread > 0, spread / (q_max - q_min), torch.ones_like(spread))
        raw_zero_point = torch.round(q_min - minimum / scale)
        # Cap above 127 to 127; cap below -128 to -128, before converting dtype.
        zero_point = raw_zero_point.clamp(q_min, q_max).to(torch.int8)
        integers = (torch.round(values / scale) + zero_point.float()).clamp(q_min, q_max).to(torch.int8)

    if granularity == "per_group":
        # Padding was only needed to compute scales; do not store fake weights.
        integers = integers.reshape(rows, -1)[:, :columns].contiguous()

    return integers, scale, zero_point

## 6. Reconstruct floating-point weights

Symmetric reconstruction is `q × scale`. Asymmetric reconstruction is `(q − zero_point) × scale`.

Convert integers and zero points to FP32 **before subtraction**; subtracting two INT8 values could overflow. Broadcasting applies the parameters to the correct tensor, row, or group. Rounding makes the reconstructed values approximate.

This work is included in inference time. We do not use a pre-built quantized operator or optimized INT8 multiplication kernel.


In [6]:
def dequantize_weight(integers, scale, granularity="per_channel", group_size=64, zero_point=None):
    """Reconstruct an FP32 matrix, broadcasting each scale over its region."""
    values = integers.to(torch.float32)
    if granularity != "per_group":
        if zero_point is not None:
            values = values - zero_point.float()
        return values * scale

    rows, columns = values.shape
    padding = (-columns) % group_size
    groups = F.pad(values, (0, padding)).reshape(rows, -1, group_size)
    if zero_point is not None:
        groups = groups - zero_point.float()
    return (groups * scale).reshape(rows, -1)[:, :columns].contiguous()

## 7. Make a usable quantized linear layer

A normal `nn.Linear` stores an FP32 weight matrix. Our `Int8Linear` stores **INT8 weights + FP32 scales + an FP32 bias**, plus INT8 zero points for asymmetric mode as buffers. Buffers are saved with the model but are not trained.

During `forward`, the layer reconstructs weights and runs ordinary `F.linear`. Activations and computation remain FP32: this is **weight-only post-training quantization** (8-bit stored weights, FP32 activations). We do not keep a permanent FP32 copy of the quantized weights.

`quantize_linears` swaps these modules into the model. We leave the final `classifier` in FP32 to protect prediction scores, and keep embeddings/normalization untouched. This limits total savings but keeps the first implementation manageable.


In [7]:
class Int8Linear(nn.Module):
    """Stores INT8 weights; reconstructs them for ordinary FP32 inference."""

    def __init__(self, linear, granularity="per_channel", group_size=64, mode="symmetric"):
        super().__init__()
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.granularity = granularity
        self.group_size = group_size
        self.mode = mode
        integers, scale, zero_point = quantize_weight(linear.weight, granularity, group_size, mode)
        # Buffers are saved in state_dict but are not trainable parameters.
        self.register_buffer("qweight", integers)
        self.register_buffer("scale", scale)
        self.register_buffer("zero_point", zero_point)
        bias = None if linear.bias is None else linear.bias.detach().float().clone()
        self.register_buffer("bias", bias)

    def reconstructed_weight(self):
        return dequantize_weight(
            self.qweight, self.scale, self.granularity, self.group_size, self.zero_point
        )

    def forward(self, inputs):
        # Do not cache a permanent FP32 copy: that would undo memory savings.
        return F.linear(inputs, self.reconstructed_weight(), self.bias)


def quantize_linears(model, granularity="per_channel", group_size=64, mode="symmetric"):
    """Replace linear layers in-place, preserving the final classifier.

    The caller should copy the FP32 model first when comparing configurations.
    """
    for name, child in list(model.named_children()):
        if isinstance(child, nn.Linear) and name != "classifier":
            setattr(model, name, Int8Linear(child, granularity, group_size, mode))
        else:
            quantize_linears(child, granularity, group_size, mode)
    return model

## 8. Try both modes and check boundary handling

The first row has small values, while the second contains an outlier. We print RMSE for both modes at all three granularities. This illustrates weight error; model accuracy is evaluated later.

The boundary example explicitly shows a zero point above 127 capped to 127, and one below −128 capped to −128. The positive-only/negative-only cases then verify the complete quantizer, including range expansion to zero. We also check that all-zero weights stay zero and padded groups preserve shape.


In [ ]:
toy = torch.tensor([[0.01, -0.02, 0.03, -0.01], [0.10, -0.20, 0.30, 2.0]])
for mode in ["symmetric", "asymmetric"]:
    for granularity in ["per_tensor", "per_channel", "per_group"]:
        q, scale, zero_point = quantize_weight(toy, granularity, group_size=2, mode=mode)
        restored = dequantize_weight(q, scale, granularity, group_size=2, zero_point=zero_point)
        rmse = (toy - restored).square().mean().sqrt().item()
        print(f"{mode}, {granularity}: {scale.numel()} scales, RMSE={rmse:.6f}")
        

raw_zero_points = torch.tensor([-383.0, 382.0])
print("Boundary example:", raw_zero_points.tolist(), "->", raw_zero_points.clamp(-128, 127).tolist())
for example in [torch.tensor([[2.0, 4.0]]), torch.tensor([[-4.0, -2.0]])]:
    q, scale, zero_point = quantize_weight(example, "per_tensor", mode="asymmetric")
    restored = dequantize_weight(q, scale, "per_tensor", zero_point=zero_point)
    assert -128 <= zero_point.item() <= 127
    assert (example - restored).abs().max() <= scale.item() * 0.51
    print("Original:", example.tolist(), "| zero point:", zero_point.item(), "| reconstructed:", restored.tolist())


symmetric, per_tensor: 1 scales, RMSE=0.004179
symmetric, per_channel: 2 scales, RMSE=0.002582
symmetric, per_group: 4 scales, RMSE=0.000396
asymmetric, per_tensor: 1 scales, RMSE=0.002487
asymmetric, per_channel: 2 scales, RMSE=0.001629
asymmetric, per_group: 4 scales, RMSE=0.000694
Boundary example: [-383.0, 382.0] -> [-128.0, 127.0]
Original: [[2.0, 4.0]] | zero point: -128 | reconstructed: [[1.992156982421875, 4.0]]
Original: [[-4.0, -2.0]] | zero point: 127 | reconstructed: [[-4.0, -1.992156982421875]]


## 9. Measure storage/error and save a reloadable checkpoint

`tensor_storage_bytes` counts stored tensor bytes, including scales, asymmetric zero points, and unquantized layers. It excludes Python objects and temporary inference memory.

`weight_rmse` compares original and reconstructed weights across **all changed linear layers**, weighting every weight equally. This is different from prediction accuracy.

`save_quantized` saves the tensors plus mapping-mode, granularity, and group-size metadata. `load_quantized` builds the architecture and our custom layers, then loads those tensors. It does not download the original FP32 weights, although this simple loader briefly allocates random FP32 layers. It therefore does not prove a peak RAM limit.


In [9]:
def tensor_storage_bytes(model):
    """Tensor payload only; excludes Python objects and temporary activations."""
    return sum(t.numel() * t.element_size() for t in model.state_dict().values())


def weight_rmse(original, quantized):
    """Global RMSE over only the linear weights changed by our quantizer."""
    originals = dict(original.named_modules())
    squared_error, count = 0.0, 0
    for name, layer in quantized.named_modules():
        if isinstance(layer, Int8Linear):
            weight = originals[name].weight.detach()
            difference = weight - layer.reconstructed_weight()
            squared_error += difference.square().sum().item()
            count += weight.numel()
    return (squared_error / count) ** 0.5 if count else 0.0


def save_quantized(model, path, granularity, group_size=64, mode="symmetric"):
    """Save only state tensors and the small amount of loader metadata."""
    torch.save(
        {
            "format_version": 2,
            "mode": mode,
            "granularity": granularity,
            "group_size": group_size,
            "state_dict": model.state_dict(),
        },
        path,
    )


def load_quantized(folder):
    """Load an exported local model folder; no pretrained FP32 download."""
    from transformers import AutoConfig, AutoModelForSequenceClassification

    folder = Path(folder)
    checkpoint = torch.load(folder / "quantized.pt", map_location="cpu", weights_only=True)
    if checkpoint["format_version"] not in {1, 2}:
        raise ValueError("Unsupported checkpoint format.")

    config = AutoConfig.from_pretrained(folder, local_files_only=True)
    # Construct the architecture, install our layer types, then fill their buffers.
    # This simple loader briefly allocates randomly initialized FP32 layers.
    model = AutoModelForSequenceClassification.from_config(
        config, attn_implementation="eager"
    ).float().cpu()
    quantize_linears(
        model, checkpoint["granularity"], checkpoint["group_size"],
        mode=checkpoint.get("mode", "symmetric"),
    )
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    return model.eval()

## 10. Measure real accuracy and CPU inference time

`inference_mode()` avoids building training gradients. Accuracy compares predicted labels with the labelled validation examples. Timing warms up the model, then reports the median of repeated forward passes on one fixed batch. The same inputs and CPU threads are used throughout.

This timing includes INT8-to-FP32 reconstruction. It can be slower than FP32. We prioritize the declared storage/accuracy constraint; speed is an additional measured trade-off.


In [10]:
@torch.inference_mode()
def evaluate_accuracy(model):
    model.eval()
    correct, total = 0, 0
    for inputs, labels in evaluation_batches:
        predictions = model(**inputs).logits.argmax(dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.numel()
    return 100.0 * correct / total


@torch.inference_mode()
def median_latency_ms(model):
    model.eval()
    for _ in range(WARMUP_RUNS):
        model(**timing_inputs)
    durations = []
    for _ in range(TIMING_RUNS):
        started = time.perf_counter()
        model(**timing_inputs)
        durations.append(1000.0 * (time.perf_counter() - started))
    return statistics.median(durations)


## 11. Evaluate and serialize the FP32 baseline

We save the baseline with `torch.save` and measure the **actual checkpoint file size**. This makes a fair comparison with quantized checkpoints saved the same way. The baseline file includes all model state tensors; tokenizer and configuration files are excluded for both cases.

The baseline accuracy is the reference for percentage-point drops. This cell may take a few minutes on a laptop CPU.


In [11]:
baseline_path = ARTIFACTS / "baseline_fp32.pt"
torch.save({"state_dict": baseline.state_dict()}, baseline_path)
baseline_bytes = baseline_path.stat().st_size
baseline_accuracy = evaluate_accuracy(baseline)
baseline_latency = median_latency_ms(baseline)

rows = [{
    "configuration": "FP32",
    "granularity": None,
    "mode": None,
    "checkpoint_mib": baseline_bytes / 1024**2,
    "tensor_mib": tensor_storage_bytes(baseline) / 1024**2,
    "reduction_percent": 0.0,
    "weight_rmse": 0.0,
    "accuracy_percent": baseline_accuracy,
    "accuracy_drop_pp": 0.0,
    "median_ms_per_batch": baseline_latency,
    "speed_ratio": 1.0,
    "meets_constraint": False,
}]
print(f"FP32: {baseline_bytes / 1024**2:.2f} MiB | accuracy {baseline_accuracy:.2f}% | {baseline_latency:.1f} ms/batch")


FP32: 255.45 MiB | accuracy 91.06% | 1216.7 ms/batch


## 12. Compare six INT8 configurations

Each run copies the **original** FP32 model, replaces the same layers, and saves a compressed checkpoint. We measure file size, RMSE, validation accuracy, and forward-pass latency. Only one quantized copy is kept at a time.

**Reduction:** `100 × (1 - quantized_bytes / FP32_bytes)`.

**Accuracy drop:** FP32 accuracy minus quantized accuracy, in **percentage points**. A negative drop means the quantized model happened to score higher on this validation set.

A configuration qualifies only if it passes **both** limits. `speed_ratio > 1` means faster than FP32; `< 1` means slower. The timer is an approximate local measurement, not a promised deployment speed.

We try **both symmetric and asymmetric**, each at all three granularities. Asymmetric zero-point metadata and subtraction overhead are included in the size and timing measurements.


In [12]:
for mode in ["symmetric", "asymmetric"]:
    for granularity in ["per_tensor", "per_channel", "per_group"]:
        print("Evaluating:", mode, granularity, flush=True)
        quantized = quantize_linears(copy.deepcopy(baseline), granularity, GROUP_SIZE, mode).eval()
        checkpoint_path = ARTIFACTS / f"int8_{mode}_{granularity}.pt"
        save_quantized(quantized, checkpoint_path, granularity, GROUP_SIZE, mode)
        checkpoint_bytes = checkpoint_path.stat().st_size
        accuracy = evaluate_accuracy(quantized)
        latency = median_latency_ms(quantized)
        reduction = 100.0 * (1.0 - checkpoint_bytes / baseline_bytes)
        accuracy_drop = baseline_accuracy - accuracy
        row = {
            "configuration": f"INT8 {mode} {granularity.replace('_', '-')}",
            "granularity": granularity,
            "mode": mode,
            "checkpoint_mib": checkpoint_bytes / 1024**2,
            "tensor_mib": tensor_storage_bytes(quantized) / 1024**2,
            "reduction_percent": reduction,
            "weight_rmse": weight_rmse(baseline, quantized),
            "accuracy_percent": accuracy,
            "accuracy_drop_pp": accuracy_drop,
            "median_ms_per_batch": latency,
            "speed_ratio": baseline_latency / latency,
            "meets_constraint": reduction >= MIN_REDUCTION_PERCENT and accuracy_drop <= MAX_ACCURACY_DROP_PP,
        }
        rows.append(row)
        print(f"  {row['checkpoint_mib']:.2f} MiB | saved {reduction:.2f}% | accuracy {accuracy:.2f}% | {latency:.1f} ms/batch", flush=True)
        del quantized
        gc.collect()


Evaluating: symmetric per_tensor
  132.28 MiB | saved 48.22% | accuracy 90.60% | 1278.3 ms/batch
Evaluating: symmetric per_channel
  132.44 MiB | saved 48.15% | accuracy 90.94% | 1374.4 ms/batch
Evaluating: symmetric per_group
  134.84 MiB | saved 47.21% | accuracy 91.06% | 1421.4 ms/batch
Evaluating: asymmetric per_tensor
  132.29 MiB | saved 48.21% | accuracy 90.37% | 1347.0 ms/batch
Evaluating: asymmetric per_channel
  132.49 MiB | saved 48.13% | accuracy 91.06% | 1547.9 ms/batch
Evaluating: asymmetric per_group
  135.50 MiB | saved 46.96% | accuracy 91.06% | 1443.1 ms/batch


## 13. Results table — include this in your report

This table contains **measured values**, not theoretical size estimates. RMSE measures weight reconstruction; accuracy measures the real sentiment task. `tensor_mib` is an additional storage measure, not peak RAM. The baseline does not meet the size reduction constraint because it has not been compressed.

The CSV preserves full precision; the notebook rounds only the display.


In [13]:
table = pd.DataFrame(rows)
table.to_csv(RESULTS / "results.csv", index=False)
columns = ["configuration", "checkpoint_mib", "reduction_percent", "weight_rmse",
           "accuracy_percent", "accuracy_drop_pp", "median_ms_per_batch", "speed_ratio", "meets_constraint"]
display(table[columns].round({"checkpoint_mib": 2, "reduction_percent": 2, "weight_rmse": 6,
                             "accuracy_percent": 2, "accuracy_drop_pp": 2,
                             "median_ms_per_batch": 1, "speed_ratio": 2}))


,configuration,checkpoint_mib,reduction_percent,weight_rmse,accuracy_percent,accuracy_drop_pp,median_ms_per_batch,speed_ratio,meets_constraint
0,FP32,255.45,0.00,0.000000,91.06,0.00,1216.7,1.00,False
1,INT8 symmetric per-tensor,132.28,48.22,0.006735,90.60,0.46,1278.3,0.95,True
2,INT8 symmetric per-channel,132.44,48.15,0.000484,90.94,0.11,1374.4,0.89,True
3,INT8 symmetric per-group,134.84,47.21,0.000276,91.06,0.00,1421.4,0.86,True
4,INT8 asymmetric per-tensor,132.29,48.21,0.004422,90.37,0.69,1347.0,0.90,True
5,INT8 asymmetric per-channel,132.49,48.13,0.000405,91.06,0.00,1547.9,0.79,True
6,INT8 asymmetric per-group,135.50,46.96,0.000242,91.06,0.00,1443.1,0.84,True


## 14. Pick a winner using the constraint

First keep only configurations with at least 45% checkpoint reduction and no more than 1.0 percentage point accuracy loss. Among those, prefer **higher observed accuracy**, then **smaller file size**, then **lower RMSE**.

This rule makes the decision explicit. Neither symmetric nor asymmetric is a preset winner. Small differences on 872 validation examples have sampling uncertainty; we are comparing engineering options on this dataset, not proving a universal ranking.

If nothing qualifies, report that outcome honestly and adjust the next experiment rather than claiming success.

### Measured decision from the completed run

**Winner: INT8 asymmetric per-channel.** It uses **132.49 MiB**, saves **48.13%**, and matches FP32 accuracy at **91.06%** (794 of 872 examples correct). Symmetric per-group and asymmetric per-group matched that accuracy too; asymmetric per-channel was the smallest of these three.

It classified one more example correctly than symmetric per-channel, with about 0.05 MiB extra storage. This small difference is specific to this validation set. Asymmetric per-tensor had lower weight RMSE but worse accuracy than symmetric per-tensor: asymmetric is not automatically better.

**Speed trade-off:** the winner took 1547.9 ms per batch versus 1216.7 ms for FP32, about **27% longer**. All six quantized configurations were slower than FP32 in this run. The target is storage reduction with limited accuracy loss, so slower inference does not disqualify them.


In [14]:
eligible = [row for row in rows if row["meets_constraint"]]
winner = min(eligible, key=lambda row: (-row["accuracy_percent"], row["checkpoint_mib"], row["weight_rmse"])) if eligible else None
if winner:
    print("Winner:", winner["configuration"])
    print(f"Saved {winner['reduction_percent']:.2f}% | accuracy drop {winner['accuracy_drop_pp']:.2f} pp")
    print(f"Speed ratio: {winner['speed_ratio']:.2f}x (above 1 = faster)")
else:
    print("No configuration meets both constraints. Keep this result in your report.")

cpu_name = platform.processor() or platform.machine()
if Path("/proc/cpuinfo").exists():
    cpu_name = next((line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines()
                     if line.startswith("model name")), cpu_name)
settings = {
    "run_date": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "model_id": MODEL_ID,
    "model_revision": getattr(baseline.config, "_commit_hash", None),
    "parameters": parameter_count,
    "dataset": "nyu-mll/glue, sst2, validation",
    "dataset_fingerprint": validation._fingerprint,
    "evaluation_examples": len(validation),
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "timing_batch_size": len(timing_inputs["input_ids"]),
    "warmup_runs": WARMUP_RUNS,
    "timing_runs": TIMING_RUNS,
    "torch_threads": torch.get_num_threads(),
    "group_size": GROUP_SIZE,
    "modes": ["symmetric", "asymmetric"],
    "minimum_reduction_percent": MIN_REDUCTION_PERCENT,
    "maximum_accuracy_drop_pp": MAX_ACCURACY_DROP_PP,
    "platform": platform.platform(),
    "cpu": cpu_name,
    "torch_version": str(torch.__version__),
    "transformers_version": transformers.__version__,
}
measurements = {"settings": settings, "results": rows, "winner": winner["configuration"] if winner else None}
(RESULTS / "measurements.json").write_text(json.dumps(measurements, indent=2))


Winner: INT8 asymmetric per-channel
Saved 48.13% | accuracy drop 0.00 pp
Speed ratio: 0.79x (above 1 = faster)


## 13. Prove that the saved model reloads and predicts

This check uses the **separate loader shipped with the model**, then compares its scores with those from the winner before export. Matching scores show that weights, scales, zero points, bias, configuration, mapping mode, and granularity were preserved.

We also print predictions on three everyday sentences. These examples help you inspect the model; they are not a substitute for the labelled accuracy evaluation.


In [ ]:
model_folder = ARTIFACTS / "hf_model"
if winner is None:
    print("Reload check skipped: no qualifying winner.")
else:
    # Rebuild the expected logits here so this cell doesn't depend on the export cell
    selected = quantize_linears(copy.deepcopy(baseline), winner["granularity"], GROUP_SIZE, winner["mode"]).eval()
    with torch.inference_mode():
        expected_logits = selected(**timing_inputs).logits.clone()
    del selected
    gc.collect()

    from quantizer import load_quantized as load_exported_model
    reloaded = load_exported_model(model_folder)
    with torch.inference_mode():
        actual_logits = reloaded(**timing_inputs).logits
    torch.testing.assert_close(actual_logits, expected_logits, rtol=1e-5, atol=1e-6)
    print("Reload check passed: exported scores match.")

    sentences = ["I really enjoyed this movie.", "The service was terrible.", "It was okay, but nothing special."]
    with torch.inference_mode():
        demo_inputs = tokenizer(sentences, padding=True, return_tensors="pt")
        labels = reloaded(**demo_inputs).logits.argmax(-1).tolist()
    for sentence, label in zip(sentences, labels):
        print(reloaded.config.id2label[label], "—", sentence)
    del reloaded
    gc.collect()

Reload check passed: exported scores match.
POSITIVE — I really enjoyed this movie.
NEGATIVE — The service was terrible.
NEGATIVE — It was okay, but nothing special.
